<div style='background:linear-gradient(135deg,#1A2E4A 0%,#0D7377 100%);padding:50px 40px;border-radius:12px;color:white;text-align:center;font-family:Arial,sans-serif;'>
  <p style='font-size:13px;letter-spacing:3px;color:#14BDBD;margin:0 0 8px 0;'>AI / ML FOUNDATIONS COHORT</p>
  <h1 style='font-size:40px;margin:0 0 8px 0;font-weight:900;'>NEURAL NETWORKS</h1>
  <h2 style='font-size:22px;font-weight:300;margin:0 0 10px 0;color:#D0D7E3;'>From Perceptron to Deep Learning — Beginner to Intermediate</h2>
  <p style='font-size:14px;color:#F0A500;font-weight:bold;margin:0 0 30px 0;'>Primary: PyTorch &nbsp;·&nbsp; Secondary: TensorFlow/Keras</p>
  <div style='width:60px;height:3px;background:#F0A500;margin:0 auto 30px auto;'></div>
  <p style='font-size:14px;color:#D0D7E3;margin:0 0 6px 0;'>Perceptron · Activations · Backprop · Training Loops · Regularisation · CNNs · Transfer Learning</p>
  <p style='font-size:13px;color:#6B8A9A;margin:0;'>Open in Google Colab (GPU recommended) or Jupyter</p>
</div>


---
## 🗺️ Session Roadmap

| Part | Topic | Framework |
|------|-------|-----------|
| A | The Biological Neuron → The Perceptron | Concept + NumPy |
| B | Building Neural Networks — Layer by Layer | PyTorch |
| C | The Training Loop — Full Manual Implementation | PyTorch |
| D | Regularisation — Dropout, Batch Norm, Weight Decay | PyTorch |
| E | Convolutional Neural Networks (CNNs) | PyTorch |
| F | Transfer Learning — Leveraging Pretrained Models | PyTorch |
| G | TensorFlow/Keras — The Same Ideas, Different API | TensorFlow |
| H | Advanced Practice Sessions | PyTorch |
| I | Capstone Project Suggestion | — |

> 🔑 **You built a neural network from scratch in the Math notebook.**  
> This notebook builds on that foundation using production frameworks at full depth.


In [ ]:
# ── Setup & GPU Check ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings, math, time
warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,
                      'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.3})

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device:          {device}')
if device.type == 'cuda':
    print(f'GPU:             {torch.cuda.get_device_name(0)}')
    print(f'VRAM:            {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('Running on CPU. For faster training, use Google Colab with GPU runtime.')
    print('Runtime → Change runtime type → T4 GPU')


---

# 🧠 Part A: From Biological Neuron to the Perceptron
#### *The atomic unit of every neural network*

---

### The Biological Neuron
A biological neuron:
1. Receives signals through **dendrites**
2. Integrates them in the **cell body** (soma)
3. If the total signal exceeds a threshold, **fires** an output through the **axon**

### The Artificial Perceptron
The perceptron maps this idea into mathematics:

$$z = \sum_{i=1}^n w_i x_i + b \qquad \text{output} = f(z)$$

- $x_i$ — inputs (features)
- $w_i$ — weights (how much each input matters)
- $b$ — bias (shifts the activation threshold)
- $f$ — activation function (the non-linearity that makes networks powerful)

### Why Non-linearity is Everything
Without activation functions, stacking layers gives you nothing more than a linear model:  
$W_3(W_2(W_1 x)) = W_{combined} x$ — it collapses to a single matrix multiplication.

Activation functions **break this collapse** — each layer learns genuinely different representations.


In [ ]:
# ── Perceptron from scratch — learning AND ────────────────────────────────────
class Perceptron:
    def __init__(self, n_inputs, lr=0.1):
        self.w  = np.random.randn(n_inputs) * 0.01
        self.b  = 0.0
        self.lr = lr

    def predict(self, x):
        return 1 if np.dot(self.w, x) + self.b >= 0 else 0

    def train(self, X, y, epochs=20):
        history = []
        for epoch in range(epochs):
            errors = 0
            for xi, yi in zip(X, y):
                pred   = self.predict(xi)
                error  = yi - pred
                self.w += self.lr * error * xi
                self.b += self.lr * error
                errors += int(error != 0)
            history.append(errors)
        return history

# AND gate
X_and = np.array([[0,0],[0,1],[1,0],[1,1]])
y_and = np.array([0, 0, 0, 1])

# OR gate
X_or  = np.array([[0,0],[0,1],[1,0],[1,1]])
y_or  = np.array([0, 1, 1, 1])

# XOR gate — a perceptron CANNOT learn this (not linearly separable)
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]])
y_xor = np.array([0, 1, 1, 0])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (X_g, y_g, name) in zip(axes, [
    (X_and,y_and,'AND'), (X_or,y_or,'OR'), (X_xor,y_xor,'XOR')
]):
    p = Perceptron(2, lr=0.1)
    hist = p.train(X_g, y_g, epochs=30)
    preds = [p.predict(x) for x in X_g]

    # Decision boundary
    if abs(p.w[1]) > 1e-6:
        x_line = np.linspace(-0.5, 1.5, 100)
        y_line = -(p.w[0]*x_line + p.b) / p.w[1]
        ax.plot(x_line, y_line, 'k--', linewidth=2, label='Decision boundary')

    colors = ['#E74C3C' if yi==0 else '#0D7377' for yi in y_g]
    ax.scatter(X_g[:,0], X_g[:,1], c=colors, s=200, zorder=5,
               edgecolors='black', linewidths=1.5)
    for xi, yi, pi in zip(X_g, y_g, preds):
        status = '✓' if yi==pi else '✗'
        ax.annotate(status, xi, textcoords='offset points', xytext=(5,5), fontsize=14)
    acc = sum(y==p for y,p in zip(y_g,preds)) / len(y_g)
    ax.set_title(f'{name} Gate — Accuracy: {acc:.0%}', fontweight='bold')
    ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.3, 1.3)
    ax.set_xlabel('Input 1'); ax.set_ylabel('Input 2')

plt.suptitle('Perceptron: AND and OR are linearly separable. XOR is NOT.',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print('XOR requires at least 2 layers — the fundamental motivation for deep networks.')


In [ ]:
# ── Why depth matters: XOR solved with a 2-layer network ─────────────────────
# A single perceptron fails on XOR. Two layers solve it.
# This is the entire reason neural networks have multiple layers.

X_xor_t = torch.FloatTensor([[0,0],[0,1],[1,0],[1,1]])
y_xor_t = torch.FloatTensor([[0],[1],[1],[0]])

class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 4)    # 2 inputs → 4 hidden neurons
        self.layer2 = nn.Linear(4, 1)    # 4 hidden → 1 output

    def forward(self, x):
        x = torch.relu(self.layer1(x))   # ReLU activation
        x = torch.sigmoid(self.layer2(x)) # Sigmoid for binary output
        return x

model   = XORNet()
opt     = optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.BCELoss()
losses  = []

for epoch in range(2000):
    opt.zero_grad()
    out  = model(X_xor_t)
    loss = loss_fn(out, y_xor_t)
    loss.backward()
    opt.step()
    losses.append(loss.item())

preds = (model(X_xor_t) >= 0.5).float()
print('=== 2-Layer Network Solves XOR ===')
print(f'Input   True  Pred  Prob')
with torch.no_grad():
    probs = model(X_xor_t)
    for xi, yi, pi, prob in zip(X_xor_t, y_xor_t, preds, probs):
        print(f'{xi.tolist()}  {int(yi.item())}     {int(pi.item())}     {prob.item():.3f}')

plt.figure(figsize=(7,3))
plt.plot(losses, color='#0D7377', linewidth=1.5)
plt.xlabel('Epoch'); plt.ylabel('BCE Loss')
plt.title('2-Layer Network Learning XOR', fontweight='bold')
plt.tight_layout(); plt.show()


---

# 🏗️ Part B: Building Neural Networks in PyTorch
#### *The right way to structure, inspect, and think about networks*

---

### The PyTorch Mental Model
In PyTorch, you define a network by subclassing `nn.Module`:
- `__init__`: define all layers and parameters
- `forward`: define how data flows through the layers
- PyTorch handles the backward pass automatically

### Network Anatomy
```
Input Layer   → receives raw features (no weights — just the shape)
Hidden Layers → learn intermediate representations
Output Layer  → final prediction (shape depends on task)
```
| Task | Output Layer | Activation | Loss |
|------|-------------|-----------|------|
| Binary Classification | 1 neuron | Sigmoid | BCELoss |
| Multi-class | N neurons (N classes) | Softmax | CrossEntropyLoss |
| Regression | 1 neuron | None (linear) | MSELoss |


In [ ]:
# ── Dataset: Heart Disease Prediction ────────────────────────────────────────
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X_raw, y_raw = make_classification(
    n_samples=1200, n_features=13, n_informative=8,
    n_redundant=3, n_classes=2, random_state=42,
    weights=[0.6, 0.4]  # slight imbalance like real medical data
)
feature_names = [
    'age','resting_bp','cholesterol','fasting_bs','rest_ecg',
    'max_hr','exercise_angina','st_depression','st_slope',
    'n_vessels','thalassemia','chest_pain_type','sex'
]

# Standardise
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X_raw)

# Split: 70% train, 15% val, 15% test
X_tv, X_test, y_tv, y_test = train_test_split(X_sc, y_raw, test_size=0.15, random_state=42, stratify=y_raw)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.176, random_state=42, stratify=y_tv)

# Convert to PyTorch tensors
def to_tensors(X, y):
    return (torch.FloatTensor(X).to(device),
            torch.FloatTensor(y).unsqueeze(1).to(device))

X_tr_t, y_tr_t = to_tensors(X_train, y_train)
X_va_t, y_va_t = to_tensors(X_val,   y_val)
X_te_t, y_te_t = to_tensors(X_test,  y_test)

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Positive class rate: {y_raw.mean():.1%}')
print(f'Features: {feature_names}')


In [ ]:
# ── Three network architectures — Shallow, Medium, Deep ──────────────────────
class ShallowNet(nn.Module):
    """1 hidden layer — simple baseline"""
    def __init__(self, n_in=13, n_h=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, n_h),
            nn.ReLU(),
            nn.Linear(n_h, 1),
            nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)


class MediumNet(nn.Module):
    """3 hidden layers with dropout"""
    def __init__(self, n_in=13):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),   nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),    nn.ReLU(),
            nn.Linear(32, 1),     nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)


class DeepNet(nn.Module):
    """5 hidden layers with residual-style skip connection"""
    def __init__(self, n_in=13):
        super().__init__()
        self.input_proj = nn.Linear(n_in, 64)
        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.Linear(64, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2)
            ) for _ in range(4)
        ])
        self.head = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, x):
        x = torch.relu(self.input_proj(x))
        for block in self.blocks:
            x = x + block(x)   # residual connection — like ResNet
        return self.head(x)


# Inspect architectures
for name, ModelClass in [('ShallowNet', ShallowNet), ('MediumNet', MediumNet), ('DeepNet', DeepNet)]:
    m = ModelClass()
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name:<15} Parameters: {n_params:>8,}')
    if name == 'MediumNet': print(m)


---

# 🔄 Part C: The Full PyTorch Training Loop
#### *Understanding every line — no magic, no black boxes*

---

The PyTorch training loop is explicit — you control every step.  
This is more verbose than Keras, but gives you complete control.

```python
for epoch in range(n_epochs):
    model.train()          # enable dropout/batchnorm training behaviour
    optimizer.zero_grad()  # clear gradients from previous step
    output = model(X)      # forward pass
    loss = loss_fn(output, y)  # compute loss
    loss.backward()        # backpropagation — compute all gradients
    optimizer.step()       # gradient descent — update all weights
```

### Optimisers — Different Ways to Descend
| Optimiser | Key Idea | When to Use |
|-----------|---------|-------------|
| **SGD** | Plain gradient descent with optional momentum | When you want full control |
| **SGD + Momentum** | Accumulates velocity, smooths oscillations | Vision tasks |
| **Adam** | Adaptive learning rates per parameter | Default choice for most tasks |
| **AdamW** | Adam with proper weight decay | Transformers, modern networks |
| **RMSprop** | Adapts lr based on recent gradient history | RNNs, non-stationary |

### Learning Rate Schedulers
A fixed learning rate is rarely optimal. Schedulers reduce lr over time:
- `StepLR` — reduce by factor every N epochs
- `CosineAnnealingLR` — cosine curve from lr_max to lr_min
- `ReduceLROnPlateau` — reduce when validation metric stops improving


In [ ]:
# ── Full production training loop with all best practices ────────────────────
def train_model(model, X_tr, y_tr, X_va, y_va,
                epochs=100, lr=1e-3, batch_size=64,
                patience=15, weight_decay=1e-4):
    """
    Full training loop with:
    - Mini-batch DataLoader
    - AdamW optimiser + weight decay
    - Cosine annealing LR scheduler
    - Early stopping
    - Best model checkpointing
    """
    model = model.to(device)
    dataset    = TensorDataset(X_tr, y_tr)
    loader     = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer  = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    loss_fn    = nn.BCELoss()

    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'lr':[]}
    best_val_loss = float('inf')
    best_weights  = None
    patience_cnt  = 0

    for epoch in range(1, epochs+1):
        # ── Training phase ──────────────────────────────────────────────────
        model.train()
        train_losses, train_correct = [], 0

        for X_b, y_b in loader:
            optimizer.zero_grad()
            out  = model(X_b)
            loss = loss_fn(out, y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
            optimizer.step()
            train_losses.append(loss.item())
            train_correct += ((out >= 0.5) == y_b.bool()).sum().item()

        # ── Validation phase ────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            val_out   = model(X_va)
            val_loss  = loss_fn(val_out, y_va).item()
            val_acc   = ((val_out >= 0.5) == y_va.bool()).float().mean().item()

        train_loss = np.mean(train_losses)
        train_acc  = train_correct / len(X_tr)
        current_lr = scheduler.get_last_lr()[0]
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)

        # ── Early stopping & checkpointing ──────────────────────────────────
        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_weights  = {k: v.clone() for k, v in model.state_dict().items()}
            patience_cnt  = 0
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f'  Early stopping at epoch {epoch}')
                break

        if epoch % 20 == 0 or epoch == 1:
            print(f'  Epoch {epoch:>4} | Train Loss: {train_loss:.4f} Acc: {train_acc:.2%}'
                  f' | Val Loss: {val_loss:.4f} Acc: {val_acc:.2%} | LR: {current_lr:.2e}')

    if best_weights:
        model.load_state_dict(best_weights)   # restore best checkpoint
    return model, history


In [ ]:
# ── Train and compare all three architectures ─────────────────────────────────
results = {}
for name, ModelClass in [('Shallow', ShallowNet), ('Medium', MediumNet), ('Deep', DeepNet)]:
    print(f'\nTraining {name}Net...')
    model = ModelClass().to(device)
    trained, hist = train_model(model, X_tr_t, y_tr_t, X_va_t, y_va_t,
                                 epochs=150, lr=1e-3, batch_size=64, patience=20)
    trained.eval()
    with torch.no_grad():
        test_preds = (trained(X_te_t) >= 0.5).cpu().numpy()
        test_acc   = (test_preds.flatten() == y_test).mean()
    results[name] = {'model': trained, 'history': hist, 'test_acc': test_acc}
    print(f'  → Final test accuracy: {test_acc:.2%}')


In [ ]:
# ── Visualise training histories ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette   = {'Shallow':'#0D7377','Medium':'#F0A500','Deep':'#E74C3C'}

for name, res in results.items():
    hist = res['history']
    ep   = range(1, len(hist['train_loss'])+1)
    c    = palette[name]
    axes[0].plot(ep, hist['train_loss'], color=c, linewidth=2, label=f'{name} train')
    axes[0].plot(ep, hist['val_loss'],   color=c, linewidth=2, linestyle='--')
    axes[1].plot(ep, hist['train_acc'],  color=c, linewidth=2, label=f'{name} train')
    axes[1].plot(ep, hist['val_acc'],    color=c, linewidth=2, linestyle='--')
    axes[2].plot(ep, hist['lr'],         color=c, linewidth=2, label=name)

axes[0].set_title('Loss (solid=train, dashed=val)', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[1].set_title('Accuracy (solid=train, dashed=val)', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[2].set_title('Learning Rate (Cosine Annealing)', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
for ax in axes: ax.legend(fontsize=8)

plt.suptitle('Training Dynamics: Shallow vs Medium vs Deep', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('\n=== Final Test Accuracy Comparison ===')
for name, res in results.items():
    print(f'  {name}Net: {res["test_acc"]:.2%}')


---

# 🛡️ Part D: Regularisation Techniques
#### *Preventing overfitting in neural networks*

---

Neural networks with millions of parameters can memorise training data perfectly.  
Regularisation techniques constrain the model to **generalise rather than memorise**.

### Dropout
During each training step, randomly **zero out** a fraction of neurons.  
This forces the network to learn redundant representations — no neuron can rely on another being present.
$$\tilde{h}_i = h_i \cdot \text{Bernoulli}(1-p) / (1-p)$$
At inference time, all neurons are active and no scaling is needed (PyTorch handles this automatically).

### Batch Normalisation
Normalises the output of each layer to have zero mean and unit variance *within each mini-batch*:
$$\hat{x} = \frac{x - \mu_{batch}}{\sigma_{batch} + \epsilon} \cdot \gamma + \beta$$
This stabilises training, allows higher learning rates, and acts as mild regularisation.

### Weight Decay (L2 Regularisation)
Adds a penalty to the loss for large weights: $L_{total} = L_{task} + \lambda \sum w^2$  
In PyTorch, pass `weight_decay=1e-4` to the optimiser.

### Early Stopping
Stop training when validation loss stops improving. Implemented in our training loop above.


In [ ]:
# ── Visualising Dropout: train vs eval mode behaviour ────────────────────────
torch.manual_seed(42)
dropout_layer = nn.Dropout(p=0.5)
x_demo = torch.ones(1, 10)

print('=== Dropout Behaviour ===')
print(f'Input:  {x_demo.numpy()}')
print('\nTraining mode (neurons randomly zeroed):')
dropout_layer.train()
for i in range(3):
    out = dropout_layer(x_demo)
    print(f'  Run {i+1}: {out.detach().numpy()}  (remaining: {(out>0).sum().item()}/10)')

print('\nEval mode (all neurons active, no zeroing):')
dropout_layer.eval()
out = dropout_layer(x_demo)
print(f'  Output: {out.detach().numpy()}')
print('\nKey: always call model.eval() before inference — it disables dropout.')


In [ ]:
# ── Dropout rate effect on overfitting ───────────────────────────────────────
class DropoutNet(nn.Module):
    def __init__(self, dropout_rate=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(13, 256), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

dropout_results = {}
for rate in [0.0, 0.3, 0.5, 0.7]:
    model  = DropoutNet(rate).to(device)
    _, hist = train_model(model, X_tr_t, y_tr_t, X_va_t, y_va_t,
                           epochs=80, lr=1e-3, batch_size=64, patience=25)
    final_tr  = hist['train_acc'][-1]
    final_val = hist['val_acc'][-1]
    dropout_results[rate] = (final_tr, final_val, hist)
    print(f'Dropout={rate:.1f}: Train={final_tr:.2%}  Val={final_val:.2%}  '
          f'Gap={final_tr-final_val:.2%}')
print('\nHigher dropout → smaller train-val gap (less overfitting)')
print('Too high → underfitting (both train and val suffer)')


In [ ]:
# ── Batch Normalisation: Training stability ───────────────────────────────────
class NetNoBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(13,128), nn.ReLU(),
            nn.Linear(128,64), nn.ReLU(),
            nn.Linear(64,1),   nn.Sigmoid())
    def forward(self, x): return self.net(x)

class NetWithBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(13,128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128,64), nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Linear(64,1),   nn.Sigmoid())
    def forward(self, x): return self.net(x)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ModelClass, label, color in [
    (NetNoBN,   'Without BatchNorm', '#E74C3C'),
    (NetWithBN, 'With BatchNorm',    '#0D7377')
]:
    m = ModelClass().to(device)
    # Train with higher lr to show instability without BN
    _, h = train_model(m, X_tr_t, y_tr_t, X_va_t, y_va_t,
                        epochs=100, lr=5e-3, batch_size=32, patience=30)
    ep = range(1, len(h['train_loss'])+1)
    axes[0].plot(ep, h['train_loss'], color=color, linewidth=2, label=label)
    axes[0].plot(ep, h['val_loss'],   color=color, linewidth=2, linestyle='--')
    axes[1].plot(ep, h['val_acc'],    color=color, linewidth=2, label=label)

axes[0].set_title('Loss: With vs Without Batch Normalisation', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[1].set_title('Val Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()


---

## ⏱️ Practice Session 1 — *Architecture Search — Find the Best Network*

> **20 minutes** — implement, train, evaluate, and interpret.

---


**Task:** Design and train the best-performing network you can on the heart disease dataset.

**Constraints and requirements:**
- Must use at least 3 hidden layers
- Must include BatchNorm AND Dropout
- Must implement gradient clipping
- Must use a learning rate scheduler (your choice)
- Must report: test accuracy, precision, recall, F1, and a confusion matrix
- Must plot: training/validation loss and accuracy curves
- Must save the best model with `torch.save()`

**Bonus challenges:**
- Implement label smoothing in your loss function
- Try `nn.LeakyReLU` or `nn.GELU` instead of ReLU
- Try different weight initialisations: `nn.init.kaiming_normal_` vs `nn.init.xavier_uniform_`


In [ ]:
# ✏️  YOUR TURN

# from sklearn.metrics import classification_report, ConfusionMatrixDisplay
# 
# class MyBestNet(nn.Module):
#     def __init__(self):
#         super().__init__()
#         # Design your architecture here
#         pass
#     def forward(self, x):
#         pass
# 
# model = MyBestNet().to(device)
# trained, hist = train_model(model, X_tr_t, y_tr_t, X_va_t, y_va_t, ...)
# 
# # Evaluate on test set
# # Plot curves
# # Save: torch.save(trained.state_dict(), 'best_model.pth')

# ── Write below ──────────────────────────────────────────────


---

# 🖼️ Part E: Convolutional Neural Networks (CNNs)
#### *Learning spatial patterns in images and sequences*

---

CNNs exploit the **spatial structure** of images using convolutional filters —  
small learnable matrices that slide across the input detecting local patterns.

### The Core Operations
| Layer | What It Does | Parameters |
|-------|-------------|------------|
| **Conv2d** | Applies learnable filters — detects edges, textures, shapes | kernel_size, out_channels, padding |
| **BatchNorm2d** | Normalises feature maps | — |
| **ReLU** | Non-linearity | — |
| **MaxPool2d** | Downsamples — keeps strongest activation in each region | kernel_size, stride |
| **AdaptiveAvgPool** | Pools to fixed size regardless of input | output_size |
| **Flatten** | Converts 2D feature maps to 1D for fully connected layers | — |
| **Linear** | Final classification head | — |

### CNN Hierarchy of Features
```
Layer 1: detects edges, corners, colour gradients
Layer 2: detects textures, simple patterns
Layer 3+: detects object parts, complex shapes
Deep layers: detects full objects, semantic concepts
```

### Receptive Field
Each neuron in a deep CNN sees a larger portion of the original input — called its **receptive field**.  
Deeper networks can capture larger patterns because their receptive field spans more of the image.


In [ ]:
# ── CNN on MNIST — handwritten digit classification ──────────────────────────
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))   # MNIST mean and std
])

try:
    train_dataset = datasets.MNIST('./data', train=True,  download=True, transform=transform)
    test_dataset  = datasets.MNIST('./data', train=False, download=True, transform=transform)
    train_loader  = DataLoader(train_dataset, batch_size=128, shuffle=True,  num_workers=0)
    test_loader   = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=0)
    print(f'MNIST loaded: {len(train_dataset):,} train  {len(test_dataset):,} test')
    print(f'Image shape: {train_dataset[0][0].shape}  Classes: {train_dataset.classes}')

    # Visualise some samples
    fig, axes = plt.subplots(2, 8, figsize=(14, 4))
    for i, ax in enumerate(axes.flatten()):
        img, label = train_dataset[i]
        ax.imshow(img.squeeze(), cmap='gray')
        ax.set_title(str(label)); ax.axis('off')
    plt.suptitle('MNIST Samples', fontweight='bold')
    plt.tight_layout(); plt.show()
    MNIST_LOADED = True
except:
    print('Could not load MNIST. Using synthetic data for CNN demonstration.')
    MNIST_LOADED = False


In [ ]:
# ── CNN Architecture ─────────────────────────────────────────────────────────
class ConvNet(nn.Module):
    """
    A solid CNN for MNIST (28×28 grayscale, 10 classes).
    Architecture: Conv→BN→ReLU→Pool × 2 → Flatten → FC → Dropout → FC
    """
    def __init__(self, n_classes=10):
        super().__init__()
        # Feature extraction — convolutional blocks
        self.features = nn.Sequential(
            # Block 1: 1×28×28 → 32×14×14
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),

            # Block 2: 32×14×14 → 64×7×7
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, n_classes)   # no softmax — CrossEntropyLoss includes it
        )

    def forward(self, x): return self.classifier(self.features(x))

cnn = ConvNet().to(device)
n_params = sum(p.numel() for p in cnn.parameters())
print(f'CNN parameters: {n_params:,}')
print(cnn)

# Test forward pass shape
dummy = torch.zeros(4, 1, 28, 28).to(device)
out   = cnn(dummy)
print(f'\nInput shape:  {dummy.shape}')
print(f'Output shape: {out.shape}  ← (batch_size, n_classes)')


In [ ]:
# ── Train the CNN ─────────────────────────────────────────────────────────────
if MNIST_LOADED:
    cnn        = ConvNet().to(device)
    optimizer  = optim.AdamW(cnn.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler  = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=1e-2,
        steps_per_epoch=len(train_loader), epochs=10
    )
    loss_fn    = nn.CrossEntropyLoss(label_smoothing=0.1)

    cnn_history = {'train_loss':[], 'train_acc':[], 'test_acc':[]}

    for epoch in range(1, 11):
        cnn.train()
        total_loss, correct, total = 0, 0, 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            out  = cnn(X_b)
            loss = loss_fn(out, y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(cnn.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
            correct    += (out.argmax(1) == y_b).sum().item()
            total      += len(y_b)

        cnn.eval()
        test_correct, test_total = 0, 0
        with torch.no_grad():
            for X_b, y_b in test_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                out = cnn(X_b)
                test_correct += (out.argmax(1) == y_b).sum().item()
                test_total   += len(y_b)

        tr_loss = total_loss / len(train_loader)
        tr_acc  = correct / total
        te_acc  = test_correct / test_total
        cnn_history['train_loss'].append(tr_loss)
        cnn_history['train_acc'].append(tr_acc)
        cnn_history['test_acc'].append(te_acc)
        print(f'Epoch {epoch:>2}/10 | Loss: {tr_loss:.4f} | Train: {tr_acc:.2%} | Test: {te_acc:.2%}')

    torch.save(cnn.state_dict(), '/tmp/cnn_mnist.pth')
    print(f'\nModel saved. Final test accuracy: {te_acc:.2%}')
else:
    print('Skipping CNN training — MNIST not loaded.')


In [ ]:
# ── Visualise: what filters and predictions look like ─────────────────────────
if MNIST_LOADED:
    cnn.eval()
    fig, axes = plt.subplots(2, 5, figsize=(14, 6))

    # Row 1: correct predictions
    # Row 2: incorrect predictions
    correct_shown, wrong_shown = 0, 0
    with torch.no_grad():
        for img, label in test_dataset:
            out   = cnn(img.unsqueeze(0).to(device))
            pred  = out.argmax(1).item()
            conf  = torch.softmax(out, dim=1).max().item()
            if pred == label and correct_shown < 5:
                ax = axes[0][correct_shown]
                ax.imshow(img.squeeze(), cmap='gray')
                ax.set_title(f'True:{label} Pred:{pred}\n{conf:.0%}', color='green', fontsize=9)
                ax.axis('off'); correct_shown += 1
            elif pred != label and wrong_shown < 5:
                ax = axes[1][wrong_shown]
                ax.imshow(img.squeeze(), cmap='gray')
                ax.set_title(f'True:{label} Pred:{pred}\n{conf:.0%}', color='red', fontsize=9)
                ax.axis('off'); wrong_shown += 1
            if correct_shown == 5 and wrong_shown == 5: break

    axes[0][0].set_ylabel('Correct', fontsize=11, fontweight='bold', color='green')
    axes[1][0].set_ylabel('Wrong',   fontsize=11, fontweight='bold', color='red')
    plt.suptitle('CNN Predictions on MNIST Test Set', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()


---

# 🔄 Part F: Transfer Learning
#### *Stand on the shoulders of giants*

---

Training a deep CNN from scratch requires:
- Millions of labelled images
- Days or weeks of GPU time
- Significant engineering expertise

**Transfer learning** lets you skip most of that.  
You take a model trained on a massive dataset (e.g. ImageNet — 14 million images, 1000 classes)  
and adapt it to your task by replacing only the final classification head.

The early layers (which detect edges, textures, basic shapes) are **universal** —  
they transfer to almost any visual task.

### Two Strategies
| Strategy | What You Do | When |
|----------|------------|------|
| **Feature extraction** | Freeze all pretrained layers, train only the head | Small dataset, similar domain |
| **Fine-tuning** | Unfreeze some or all layers, train end-to-end with low LR | Larger dataset, different domain |

### Available Pretrained Models in torchvision
`ResNet18/50/101`, `EfficientNet`, `ViT`, `DenseNet`, `MobileNet`, `ConvNeXt`, and many more.


In [ ]:
# ── Transfer Learning with ResNet18 ──────────────────────────────────────────
from torchvision import models

# Load pretrained ResNet18
resnet = models.resnet18(weights='IMAGENET1K_V1')

print('=== ResNet18 Architecture Summary ===')
print(f'Total parameters: {sum(p.numel() for p in resnet.parameters()):,}')
print(f'Trainable:        {sum(p.numel() for p in resnet.parameters() if p.requires_grad):,}')
print(f'\nFinal layer: {resnet.fc}')

# ── Strategy 1: Feature Extraction — freeze everything except the head ─────────
for param in resnet.parameters():
    param.requires_grad = False

# Replace the final layer for binary classification
n_features = resnet.fc.in_features
resnet.fc  = nn.Sequential(
    nn.Linear(n_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 1),
    nn.Sigmoid()
)

trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
total     = sum(p.numel() for p in resnet.parameters())
print(f'\nAfter freezing:')
print(f'  Trainable parameters: {trainable:,}  ({trainable/total:.1%} of total)')
print(f'  Frozen parameters:    {total-trainable:,}')
print(f'\nOnly training the head — backbone acts as a fixed feature extractor.')


In [ ]:
# ── Strategy 2: Fine-tuning — progressive unfreezing ─────────────────────────
resnet2 = models.resnet18(weights='IMAGENET1K_V1')

# Freeze all first
for param in resnet2.parameters():
    param.requires_grad = False

# Unfreeze the last 2 residual blocks + head (layer3, layer4, fc)
for name, param in resnet2.named_parameters():
    if any(layer in name for layer in ['layer3','layer4','fc']):
        param.requires_grad = True

resnet2.fc = nn.Sequential(
    nn.Linear(resnet2.fc.in_features, 256), nn.ReLU(), nn.Dropout(0.4),
    nn.Linear(256, 1), nn.Sigmoid()
)

print('=== Progressive Unfreezing Strategy ===')
print(f'{'Layer':<30} {'Trainable'}')
print('-'*42)
seen = set()
for name, param in resnet2.named_parameters():
    layer_name = name.split('.')[0]
    if layer_name not in seen:
        seen.add(layer_name)
        status = '✅ Trainable' if param.requires_grad else '❄️  Frozen'
        print(f'{layer_name:<30} {status}')

trainable2 = sum(p.numel() for p in resnet2.parameters() if p.requires_grad)
total2     = sum(p.numel() for p in resnet2.parameters())
print(f'\nTrainable: {trainable2:,} / {total2:,}  ({trainable2/total2:.1%})')
print('Fine-tuning the last blocks allows domain adaptation while keeping general features.')


---

# 🟠 Part G: TensorFlow / Keras
#### *The same concepts, a different (and often simpler) API*

---

TensorFlow with the Keras API takes a different philosophy from PyTorch:  
**high-level simplicity by default**, with the ability to go low-level when needed.

### PyTorch vs TensorFlow/Keras — Key Differences
| Aspect | PyTorch | TensorFlow/Keras |
|--------|---------|------------------|
| Training loop | Manual (explicit) | Automatic via `.fit()` |
| Debugging | Easy — Python native | Can be harder |
| Research use | Dominant | Less common now |
| Production/mobile | Via TorchServe | TF Serving, TF Lite |
| Graph execution | Eager by default | Eager + `@tf.function` |
| Community trend | Growing fast | Still very large |

**Rule of thumb:** Learn PyTorch deeply (as we have). Know Keras well enough to read  
and use it — most tutorials, courses, and deployed models you encounter will use one or the other.


In [ ]:
# ── Same heart disease problem in TensorFlow/Keras ───────────────────────────
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, callbacks
    print(f'TensorFlow version: {tf.__version__}')
    print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')
    TF_AVAILABLE = True
except ImportError:
    print('TensorFlow not installed. pip install tensorflow')
    print('Pre-installed on Google Colab.')
    TF_AVAILABLE = False


In [ ]:
if TF_AVAILABLE:
    tf.random.set_seed(42)

    # ── Build model using Keras Functional API ────────────────────────────────
    inputs = keras.Input(shape=(13,), name='features')
    x = layers.Dense(128, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid', name='prediction')(x)

    tf_model = keras.Model(inputs=inputs, outputs=outputs, name='HeartDiseaseNet')
    tf_model.summary()

    # ── Compile — specify optimiser, loss, and metrics ────────────────────────
    tf_model.compile(
        optimizer = keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
        loss      = 'binary_crossentropy',
        metrics   = ['accuracy', keras.metrics.AUC(name='auc'),
                     keras.metrics.Precision(name='precision'),
                     keras.metrics.Recall(name='recall')]
    )

    # ── Callbacks — Keras equivalent of our manual early stopping + scheduler ──
    cb = [
        callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6),
        callbacks.ModelCheckpoint('/tmp/tf_best.keras', save_best_only=True, monitor='val_loss')
    ]

    # ── Train — single line ────────────────────────────────────────────────────
    history_tf = tf_model.fit(
        X_train, y_train,
        validation_data = (X_val, y_val),
        epochs          = 150,
        batch_size      = 64,
        callbacks       = cb,
        verbose         = 0
    )

    # ── Evaluate ──────────────────────────────────────────────────────────────
    results_tf = tf_model.evaluate(X_test, y_test, verbose=0)
    print('\n=== TensorFlow/Keras Results ===')
    for name, val in zip(tf_model.metrics_names, results_tf):
        print(f'  {name:<12}: {val:.4f}')

    # ── Plot history ──────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_tf.history['loss'],     color='#0D7377', label='Train')
    axes[0].plot(history_tf.history['val_loss'], color='#F0A500', label='Val')
    axes[0].set_title('Keras: Loss Curve', fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()
    axes[1].plot(history_tf.history['accuracy'],     color='#0D7377', label='Train')
    axes[1].plot(history_tf.history['val_accuracy'], color='#F0A500', label='Val')
    axes[1].set_title('Keras: Accuracy Curve', fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend()
    plt.tight_layout(); plt.show()


In [ ]:
if TF_AVAILABLE:
    # ── PyTorch vs Keras: Side by side comparison ─────────────────────────────
    # Get best PyTorch model results
    best_pt_name = max(results, key=lambda k: results[k]['test_acc'])
    best_pt_acc  = results[best_pt_name]['test_acc']

    print('=== Framework Comparison on Heart Disease Dataset ===')
    print(f'  Best PyTorch ({best_pt_name}Net):  {best_pt_acc:.2%}')
    print(f'  TensorFlow/Keras:       {results_tf[1]:.2%}')
    print(f'\n  Same architecture, same data — small differences come from')
    print(f'  random initialisation and optimiser state, not framework quality.')
    print(f'\n  Both frameworks are production-grade. Pick one to master.')


---

## ⏱️ Practice Session 2 — *Advanced Practice — Multi-class + Custom Loss + Model Analysis*

> **25 minutes** — implement, train, evaluate, and interpret.

---


This is a more demanding practice session. Work through each part systematically.

### Part H1 — Multi-class Classification
Use `sklearn.datasets.load_digits()` (10-class digit recognition, 64 features).
- Build a network that achieves >97% test accuracy
- Use `nn.CrossEntropyLoss` (not BCE — this is multi-class)
- Output layer: `nn.Linear(?, 10)` — no softmax (CrossEntropyLoss includes it)
- Plot a confusion matrix for all 10 digit classes
- Report per-class precision and recall

### Part H2 — Custom Loss Function
Implement **Focal Loss** — a loss designed for class imbalance (used in object detection):
$$FL(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$
- Apply it to the heart disease dataset
- Try gamma values: 0 (reduces to BCE), 1, 2, 5
- Does Focal Loss improve recall on the minority class?

### Part H3 — Model Analysis & Interpretability
- Plot the weight distribution of your trained network across all layers
- Visualise the activations of hidden neurons on 5 test samples
- Identify the 5 most influential input features using permutation importance
- Does the network agree with domain knowledge about heart disease risk factors?


In [ ]:
# ✏️  YOUR TURN

# # H1 — Multi-class on digits
# from sklearn.datasets import load_digits
# from sklearn.metrics import ConfusionMatrixDisplay
# 
# digits = load_digits()
# X_dg, y_dg = digits.data, digits.target
# # Normalise, split, convert to tensors
# # Build network with output size 10
# # Train with nn.CrossEntropyLoss()
# # Evaluate and plot confusion matrix
# 
# # H2 — Focal Loss
# class FocalLoss(nn.Module):
#     def __init__(self, alpha=0.25, gamma=2.0):
#         super().__init__()
#         self.alpha = alpha
#         self.gamma = gamma
#     def forward(self, pred, target):
#         bce  = F.binary_cross_entropy(pred, target, reduction='none')
#         pt   = torch.exp(-bce)
#         # Focal: alpha * (1-pt)^gamma * BCE
#         loss = self.alpha * (1 - pt)**self.gamma * bce
#         return loss.mean()
# 
# # H3 — Weight distribution
# # fig, axes — one subplot per layer
# # for name, param in model.named_parameters():
# #     if 'weight' in name: axes[i].hist(param.data.cpu().numpy().flatten())

# ── Write below ──────────────────────────────────────────────


---

# 🚀 Part I: Capstone Project Suggestion
#### *A real problem, a real impact*

---

## Capstone 1 — Retinal Disease Detection from Fundus Images

### The Problem
Diabetic retinopathy is the leading cause of preventable blindness worldwide.  
Early detection is critical — but the shortage of trained ophthalmologists in  
low-resource settings means millions of patients are never screened.

An accurate, fast, and cheap AI-powered screening tool could save millions of people's sight.

This is not a hypothetical problem — it is exactly what Google's DeepMind built,  
and what multiple African health-tech startups are building right now.

---

### Your Mission
Build a **convolutional neural network** that classifies retinal fundus images  
into severity levels of diabetic retinopathy:

| Class | Label | Description |
|-------|-------|-------------|
| 0 | No DR | No disease |
| 1 | Mild | Minor microaneurysms |
| 2 | Moderate | More lesions, may affect vision |
| 3 | Severe | Extensive lesions, high risk |
| 4 | Proliferative DR | Most advanced — immediate treatment needed |

---

### Dataset
Use the **APTOS 2019 Blindness Detection** dataset from Kaggle:
[kaggle.com/c/aptos2019-blindness-detection](https://www.kaggle.com/c/aptos2019-blindness-detection)

- 3,662 training images + labels
- 1,928 test images
- High-resolution fundus photographs
- Real clinical data from rural India

---

### Technical Requirements

**Minimum (Pass):**
- Load and preprocess the fundus images (resize, normalise, augment)
- Train a CNN from scratch or use transfer learning
- Achieve >75% validation accuracy on the 5-class problem
- Report per-class metrics — recall matters most for classes 3 and 4
- Plot training curves and a confusion matrix

**Intermediate (Good):**
- Use transfer learning with EfficientNet or ResNet50
- Apply data augmentation: random flips, rotations, colour jitter, cutout
- Handle class imbalance: weighted loss or oversampling
- Use a regression approach (predict 0–4 as continuous) then round — often works better
- Achieve >82% validation accuracy or QWK (Quadratic Weighted Kappa) > 0.85

**Advanced (Excellent):**
- Implement test-time augmentation (TTA) for better predictions
- Use a model ensemble (multiple architectures averaged)
- Generate Grad-CAM visualisations showing which parts of the retina triggered classification
- Achieve QWK > 0.90 (competitive Kaggle score)
- Write a clinical interpretation: what would this model mean for a rural screening programme?

---

### Starter Architecture Suggestion
```python
# Feature extractor: EfficientNet-B3 pretrained on ImageNet
# Input: 300×300 RGB fundus images
# Head: GlobalAvgPool → Dense(512, ReLU) → Dropout(0.4) → Dense(5, Softmax)
# Loss: CrossEntropyLoss with class weights
# Optimiser: AdamW with cosine annealing
# Augmentation: RandomHFlip, RandomRotation(±15°), ColorJitter, RandomResizedCrop
```

---

### Alternative Capstone Options
If you prefer a different domain, these are equally challenging and impactful:

**Option B — Audio-based Malaria Detection**  
Build a classifier that detects malaria from the sound of wingbeats using spectrograms + CNN.  
Dataset: [github.com/HumBug-Mosquito/HumBugDB](https://github.com/HumBug-Mosquito/HumBugDB)

**Option C — Chest X-Ray Pathology Detection**  
Multi-label classification of 14 thoracic diseases from chest X-rays.  
Dataset: NIH ChestX-ray14 (available on Kaggle)

**Option D — Crop Disease Detection**  
Identify plant diseases from leaf photographs — highly relevant in African agriculture.  
Dataset: PlantVillage on Kaggle or TensorFlow Datasets

---

### Evaluation Rubric
| Criterion | Weight |
|-----------|--------|
| Data loading, preprocessing, and augmentation | 15% |
| Model architecture design and justification | 20% |
| Training loop quality (callbacks, scheduling, monitoring) | 15% |
| Performance metrics and honest evaluation | 20% |
| Handling class imbalance | 10% |
| Visualisations (training curves, confusion matrix, Grad-CAM) | 10% |
| Written interpretation and clinical/business framing | 10% |

> 💡 **The best capstone projects are not just technically correct —  
> they answer the question: 'What would this mean in the real world?'**


---
## ✅ What You Covered in This Notebook
| Concept | Depth |
|---------|-------|
| Perceptron — biological to mathematical | Deep |
| Why XOR requires depth — the key motivation for deep networks | Deep |
| PyTorch: nn.Module, Sequential, Functional API | Deep |
| Full training loop: DataLoader, optimiser, scheduler, early stopping, checkpointing | Deep |
| Gradient clipping, weight decay, AdamW | Deep |
| Dropout, Batch Normalisation, residual connections | Deep |
| CNNs: Conv2d, MaxPool, feature extraction, classification head | Deep |
| Transfer Learning: feature extraction vs fine-tuning, progressive unfreezing | Deep |
| TensorFlow/Keras: Functional API, callbacks, compile, fit | Solid |
| Multi-class classification, Focal Loss, model analysis | Advanced practice |
| Capstone: Retinal disease detection — real clinical impact | Project |

<div style='background:linear-gradient(135deg,#1A2E4A 0%,#0D7377 100%);padding:35px;border-radius:10px;color:white;text-align:center;'>
  <h3 style='margin:0 0 10px 0;color:#14BDBD;'>You now build networks, not just run them.</h3>
  <p style='color:#D0D7E3;margin:0 0 10px 0;'>The path from here leads to transformers, diffusion models, and systems that change the world.</p>
  <p style='color:#F0A500;font-weight:bold;margin:0;'>Build the capstone. It will be the first line of your ML portfolio. 🚀</p>
</div>
